# Ensemble Pipeline — Executive Summary

**Project:** AI4EAC Liquidity Stress Early Warning Challenge (Zindi Africa)  
**Notebook:** `09_ensemble.ipynb`  
**Author:** Henry  
**Date:** April 2026  
**Stage:** Phase B — Ensemble pipeline execution and strategy selection  

---

## 1. Objective

Build and evaluate a four-strategy ensemble pipeline using the three Platt-calibrated
base model OOF predictions from `08_multi_model_analysis.ipynb`. Select the optimal
deployment strategy for the inference pipeline based on OOF performance and
generalisation risk assessment.

---

## 2. Ensemble Results

| Strategy | Log Loss | AUC | Score | Brier |
|---|---|---|---|---|
| **Optimised weighted average** | **0.25607** | 0.90500 | **0.19164** | 0.07639 |
| Simple average | 0.25630 | **0.90507** | 0.19175 | 0.07642 |
| XGBoost (individual) | 0.25835 | 0.90378 | 0.19350 | 0.07719 |
| CatBoost (individual) | 0.25865 | 0.90223 | 0.19430 | 0.07736 |
| LightGBM (individual) | 0.26117 | 0.90284 | 0.19557 | 0.07775 |
| Stacking | 0.26603 | 0.90362 | 0.19817 | 0.07756 |
| Stacking calibrated | 0.27790 | 0.89877 | 0.20723 | 0.07977 |

Optimised ensemble weights: LightGBM=0.195, XGBoost=0.401, CatBoost=0.404

---

## 3. Key Findings

### 3.1 Stacking Underperformed Averaging — Root Cause Analysis

The most technically significant finding is that the stacking meta-model ranked
last among all ensemble strategies. This is a principled, expected result given
the inter-model correlation structure identified in notebook 08.

**Root cause: multicollinearity in meta-features.** With pairwise correlations of
0.962–0.978 between the three base models, the LogisticRegression meta-model cannot
learn reliable conditional weights. In near-singular input spaces, coefficient estimates
become numerically unstable regardless of regularisation strength — the optimiser assigns
extreme weights to break degeneracy rather than reflecting true model quality.

**Evidence 1 — Monotone fold degradation.** Stacking fold scores deteriorated
systematically from 0.18892 (fold 0) to 0.20437 (fold 3), with ±0.00613 standard
deviation — 3× larger than any individual base model's fold variance on the same data.
This is the fingerprint of meta-model instability, not random sampling variance.

**Evidence 2 — Coefficient inversion.** The meta-model assigned CatBoost the highest
coefficient (2.809) despite it being the weakest model pre-calibration. The coefficient
ordering (CatBoost > LightGBM > XGBoost) contradicts the individual model quality
ordering (XGBoost > CatBoost > LightGBM by Log Loss). When a meta-model disagrees
with direct evidence about model quality, it is fitting noise.

**Evidence 3 — Calibrated stacking deteriorated further.** Applying Platt calibration
to the stacking output raised Log Loss from 0.26603 to 0.27790 — a 0.012 increase.
Calibration applied to a collinearity-damaged output amplifies rather than corrects
the error, confirming the stacking foundation was unstable.

### 3.2 Weighted Average vs Simple Average — The 0.00011 Gap

The optimised weighted average (0.25607) beat the simple average (0.25630) by only
0.00011 Log Loss. This gap is smaller than the variance introduced by test set
distribution shift and will not reliably reproduce on unseen data. The optimised
weights (LightGBM=0.195, XGBoost=0.401, CatBoost=0.404) are OOF-fitted and will
not perfectly generalise. The simple average produces higher AUC (0.90507 vs 0.90500)
and is more robust to distribution shift.

### 3.3 Calibration Is the Dominant Lever

The cumulative improvement across all phases:

| Stage | Log Loss | Improvement |
|---|---|---|
| Raw LightGBM baseline (notebook 06) | 0.30228 | — |
| Platt-calibrated LightGBM (individual) | 0.26117 | −0.041 (89% of total gain) |
| Simple ensemble (3 calibrated models) | 0.25630 | −0.005 (11% of total gain) |
| Optimised weighted average | 0.25607 | −0.000 |

Calibration delivered 89% of the total Log Loss improvement. Ensembling delivered
the remaining 11%. This validates the strategic decision to prioritise calibration
before ensembling — the order of operations was correct.

---

## 4. Deployment Decision

**Selected strategy: Simple average of three Platt-calibrated predictions.**

**Rationale:**
- 0.00023 Log Loss gap vs optimised weights is within test set variance
- Higher AUC than optimised weights (0.90507 vs 0.90500)
- No OOF fitting — not optimistic on test data
- Simplest implementation — lowest risk in inference pipeline

**Production formula:**
```python
ensemble_pred = (lgb_platt + xgb_platt + cat_platt) / 3.0
final_pred = np.clip(ensemble_pred, 1e-6, 1 - 1e-6)
```

---

## 5. Open Diagnostic — Positive Rate Discrepancy

The OOF y_true shows a positive rate of 0.1500 (15%), while the competition brief
states ~10,500 positives in 40,000 training rows (~26%). This discrepancy requires
verification before the inference pipeline is built. Possible explanations: the 26%
figure refers to the full 70,000-row dataset across all 7 snapshots, not the 40,000
training split; or the positive rate varies across snapshot months with the training
split over-representing low-stress months. This does not affect the ensemble strategy
selection but must be resolved before `scale_pos_weight` parameters are finalised
for any retraining.

---

## 6. Path Forward for Stacking

Stacking is not discarded — it is deferred. After Phase C (Optuna tuning), the base
models will have different structural parameters and predictions with lower pairwise
correlation. Re-running the ensemble pipeline with tuned models may reduce correlation
to 0.93–0.94, at which point stacking gains become meaningful. The pipeline
architecture, meta-model design, and artifact saving are all in place for this.

Additionally, adding top raw features (`use_raw_features=True`) to the meta-feature
matrix gives the stacking model signal beyond the collinear OOF predictions. The top
5 features by cross-model SHAP agreement (from `feature_importance_combined.csv`)
are the natural candidates.

---

## 7. Artifacts Saved

```
outputs/experiments/v4_ensemble/run_20260430_074450/
    ensemble_oof.npy          calibrated stacking OOF (retained for analysis)
    stacking_oof.npy          raw stacking meta-model OOF
    ensemble_weights.json     optimised weights + model order
    meta_model.pkl            LogisticRegression meta-model
    meta_calibrator.pkl       Platt calibrator for ensemble output
    stacking_results.json     full metrics all strategies
    feature_importance.csv    meta-model coefficient analysis
    metadata.json             config + results record
```

**Inference pipeline artifact (deployment):**
```
outputs/calibration/{model}/calibrator_platt.pkl   (3 files, one per model)
```

---

## 8. Expected Leaderboard Position

| Stage | OOF Log Loss | Test Estimate |
|---|---|---|
| Raw LightGBM (baseline) | 0.302 | ~0.310 |
| Calibrated ensemble (current) | 0.256 | ~0.260–0.268 |
| After Optuna + re-ensemble | ~0.250 | ~0.255–0.263 |

Test estimates are 0.004–0.012 higher than OOF due to distribution shift
between training snapshots (Jan–Apr 2024) and test period (May–Jul 2024).

---

## 9. Next Steps

**Phase C — `src/tuning/tune.py`:** Optuna hyperparameter tuning on LightGBM
and XGBoost (100–150 trials each). Objective: composite score. Encourage structural
diversity by searching wider num_leaves and min_child_samples ranges than the
baseline configs. Target: reduce pairwise correlation to <0.95 and re-evaluate
stacking.

**Phase D — `10_shap_interpretability.ipynb`:** SHAP analysis on the final ensemble.
Global + local SHAP on the simple-average ensemble. Business narrative translation
connecting model internals to financial distress early warning signals.

**Phase F — `src/inference/inference.py`:** Build inference pipeline consuming
the three base model fold-level models and three Platt calibrators. Verify positive
rate discrepancy before finalising the pipeline.

---

*All artifacts versioned and reproducible from `configs/{model}_v2.yaml` and
`outputs/experiments/baseline/{model}/run_*/` with fixed seed=42.*